#  Amazon E-Commerce Sales Analysis
## Notebook 2 — Data Cleaning

*Objective:* Clean the raw dataset by fixing data types, 
handling missing values, removing irrelevant columns, 
and filtering invalid orders to prepare 
analysis-ready data.

*Input:* Amazon Sale Report.csv — 128,975 rows, 24 columns  
*Output:* amazon_cleaned.csv — clean, analysis-ready dataset

---

In [1]:
# ==========================================================
# STEP 1: IMPORT LIBRARIES AND LOAD DATA
# ==========================================================

import pandas as pd
import numpy as np

df = pd.read_csv('Amazon Sale Report.csv', encoding='latin1')

print(f"data loaded: {df.shape[0]} rows, {df.shape[1]} columns")

data loaded: 128975 rows, 24 columns


C:\Users\pushp\AppData\Local\Temp\ipykernel_17920\849925289.py:8: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Amazon Sale Report.csv', encoding='latin1')


In [2]:
# ==========================================================
# STEP 2: CHECK FOR MISSING VALUES
# ==========================================================

# count missng values in each column
missing = df.isnull().sum()

# calculate percentage
missing_percent = (missing / len(df)) * 100

# combine into a table
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage': missing_percent
})

# show columns with missing values
missing_df = missing_df[missing_df['Missing Count'] > 0]
print("Columns with missing values:")
print(missing_df.sort_values('Missing Percentage', ascending=False))

Columns with missing values:
                  Missing Count  Missing Percentage
fulfilled-by              89698           69.546811
promotion-ids             49153           38.110487
Unnamed: 22               49050           38.030626
currency                   7795            6.043807
Amount                     7795            6.043807
Courier Status             6872            5.328164
ship-postal-code             33            0.025586
ship-state                   33            0.025586
ship-city                    33            0.025586
ship-country                 33            0.025586


## Missing Values Analysis

| Column | Missing Count | Missing % | Action |
|--------|--------------|-----------|--------|
| fulfilled-by | 89,698 | 69.5% | Drop column |
| promotion-ids | 49,153 | 38.1% | Drop column |
| Unnamed: 22 | 49,050 | 38.0% | Drop column |
| currency | 7,795 | 6.0% | Drop rows |
| Amount | 7,795 | 6.0% | Drop rows |
| Courier Status | 6,872 | 5.3% | Drop rows |
| ship-city/state | 33 | 0.02% | Drop rows |

*Decision Rule:*
- Missing > 30% → Drop entire column
- Missing < 10% → Drop those rows only

In [3]:
# ============================================================
# STEP 3: DROP UNNECESSARY COLUMNS
# ============================================================

# Columns to drop and why:
# fulfilled-by → 69.5% missing, not useful for analysis
# promotion-ids → 38.1% missing, not relevant
# Unnamed: 22 → garbage column, no meaning
# courier-status → 5.3% missing, redundant with Status column
# ship-postal-code → postal code not needed for analysis
# ASIN → Amazon internal product code, not useful

columns_to_drop = [
    'fulfilled-by',
    'promotion-ids',
    'Unnamed: 22',
    'Courier Status',
    'ship-postal-code',
    'ASIN'
]

df.drop(columns=columns_to_drop, inplace=True)
print(f"Columns Dropped Successfully!")
print(f"Remaining Columns: {df.shape[1]}")
print(f"Column Names: {list(df.columns)}")

Columns Dropped Successfully!
Remaining Columns: 18
Column Names: ['index', 'Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel ', 'ship-service-level', 'Style', 'SKU', 'Category', 'Size', 'Qty', 'currency', 'Amount', 'ship-city', 'ship-state', 'ship-country', 'B2B']


In [4]:
# ============================================================
# STEP 4: FIX DATE COLUMN
# ============================================================

# check current dataype of date
print(f"Before: Date column type = {df['Date'].dtypes}")

# convert to datetime
df['Date'] = pd.to_datetime(df['Date'], format='%m-%d-%y')

# extract useful date parts

df['Month'] =  df['Date'].dt.month
df['Month_name'] = df['Date'].dt.month_name()
df['Year'] = df['Date'].dt.year

print(f"After: Date column type = {df['Date'].dtypes}")
print(f"\nDate range: {df['Date'].min()}  to {df['Date'].max()}")
print(f"\nNew columns added: Month, Month_name, Year")
print(f"\nSample dates:")
print(df[['Date', 'Month', 'Month_name', 'Year']].head())

Before: Date column type = object
After: Date column type = datetime64[ns]

Date range: 2022-03-31 00:00:00  to 2022-06-29 00:00:00

New columns added: Month, Month_name, Year

Sample dates:
        Date  Month Month_name  Year
0 2022-04-30      4      April  2022
1 2022-04-30      4      April  2022
2 2022-04-30      4      April  2022
3 2022-04-30      4      April  2022
4 2022-04-30      4      April  2022


In [5]:
# ============================================================
# STEP 5: HANDLE MISSING VALUES
# ============================================================

# drop rows where amount is missing
# (currency  and amount missing together - same 7795 rows)

df = df[df['Amount'].notna()]

# drop rows where ship-state is missing (33 rows)
df = df[df['ship-state'].notna()]

print(f"Missing values handeled!")
print(f"Remaining rows: {df.shape[0]}")
print(f"Remaining columns: {df.shape[1]}")

# verify no missing values remain
print(f"\nMissing values remaining:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values handeled!
Remaining rows: 121149
Remaining columns: 21

Missing values remaining:
Series([], dtype: int64)


In [6]:
# ============================================================
# STEP 6A: REMOVE INVALID ORDERS
# ============================================================

print(f"Before cleaning: {df.shape[0]} rows and {df.shape[1]} columns")

# check status

print(f"\nOrder status count:")
print(df['Status'].value_counts())

Before cleaning: 121149 rows and 21 columns

Order status count:
Status
Shipped                          77580
Shipped - Delivered to Buyer     28754
Cancelled                        10761
Shipped - Returned to Seller      1947
Shipped - Picked Up                973
Pending                            656
Pending - Waiting for Pick Up      281
Shipped - Returning to Seller      145
Shipped - Out for Delivery          35
Shipped - Rejected by Buyer         11
Shipped - Lost in Transit            5
Shipped - Damaged                    1
Name: count, dtype: int64


In [7]:
# ============================================================
# STEP 6B: KEEP ONLY VALID ORDERS
# ============================================================

valid_status = ['Shipped', 
                'Shipped - Delivered to Buyer',
                'Shipped - Picked Up',
                'Shipped - Out for Delivery']

# keep only valid orders
df = df[df['Status'].isin(valid_status)]

print(f"Invalid orders removed!")
print(f"Remaining rows: {df.shape[0]}")
print(f"\nStatus count after filtering:")
print(df['Status'].value_counts())

Invalid orders removed!
Remaining rows: 107342

Status count after filtering:
Status
Shipped                         77580
Shipped - Delivered to Buyer    28754
Shipped - Picked Up               973
Shipped - Out for Delivery         35
Name: count, dtype: int64


In [8]:
# ============================================================
# STEP 7: REMOVE ZERO AMOUNT ORDERS
# ============================================================

print(f"Before: {df.shape[0]} rows")

# check zero
zero_amount = df[df['Amount'] == 0].shape[0]
print(f"Zero amount orders found: {zero_amount} rows")

# remove zero amount orders
df = df[df['Amount'] > 0]

print(f"After: {df.shape[0]} rows")
print(f"Removed: {zero_amount} zero amount orders")

Before: 107342 rows
Zero amount orders found: 2262 rows
After: 105080 rows
Removed: 2262 zero amount orders


In [24]:
# ============================================================
# STEP 8: SAVE CLEAN DATASET
# ============================================================

# Save cleaned dataframe as new CSV
df.to_csv('amazon_cleaned.csv', index=False)

print(f"Clean dataset saved as amazon_cleaned.csv!")
print(f"Final shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nCleaning Summary:")
print(f"Original rows:     128,975")
print(f"Final clean rows:  {df.shape[0]}")
print(f"Rows removed:      {128975 - df.shape[0]}")
print(f"Data retained:    {round(df.shape[0] / 128975 * 100, 1)}%")

Clean dataset saved as amazon_cleaned.csv!
Final shape: 105080 rows, 21 columns

Cleaning Summary:
Original rows:     128,975
Final clean rows:  105080
Rows removed:      23895
Data retained:    81.5%


## Data Cleaning Summary

### What We Started With
- Raw rows: 128,975
- Raw columns: 24

### Cleaning Steps Performed
1. Dropped 6 irrelevant columns
2. Converted Date column to datetime
3. Extracted Month, Month_Name, Year from Date
4. Removed 7,828 rows with missing Amount/State
5. Removed 13,807 cancelled/invalid orders
6. Removed 2,262 zero amount orders

### Final Clean Dataset
- Clean rows: 105,080
- Clean columns: 21
- Data retained: 81.5%

### Key Business Insights Found During Cleaning
- Cancellation rate: 8.9%
- 98.5% of orders are single item purchases
- Data covers: March 2022 to June 2022
- Average order value: ₹648

### Output
- Saved as: amazon_cleaned.csv
- Ready for EDA analysis